<a href="https://colab.research.google.com/github/Gabo190594/recommender-system-supermarket/blob/main/notebooks/05_recommender_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clonar tu repositorio desde GitHub
!git clone https://github.com/Gabo190594/recommender-system-supermarket.git

# Entrar a la carpeta del proyecto
%cd recommender-system-supermarket

Cloning into 'recommender-system-supermarket'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 78 (delta 31), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 766.36 KiB | 5.40 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/recommender-system-supermarket


In [3]:
from IPython.display import Markdown, display

display(Markdown("""
# 05. Sistema de recomendación

## Objetivo
Construir un recomendador basado en similitud entre usuarios.

## ¿Qué hace este notebook?
1. Carga el dataset con la recompensa final.
2. Construye una matriz usuario-producto.
3. Calcula la similitud entre usuarios usando similitud coseno.
4. Identifica usuarios similares a cada usuario objetivo.
5. Genera recomendaciones a partir de productos consumidos por usuarios parecidos.

## ¿Qué lógica sigue?
Si dos usuarios tienen comportamientos similares, es razonable suponer que
los productos valorados positivamente por uno también podrían interesar al otro.

## Resultado esperado
Obtener una lista de productos recomendados para un usuario dado.
"""))


# 05. Sistema de recomendación

## Objetivo
Construir un recomendador basado en similitud entre usuarios.

## ¿Qué hace este notebook?
1. Carga el dataset con la recompensa final.
2. Construye una matriz usuario-producto.
3. Calcula la similitud entre usuarios usando similitud coseno.
4. Identifica usuarios similares a cada usuario objetivo.
5. Genera recomendaciones a partir de productos consumidos por usuarios parecidos.

## ¿Qué lógica sigue?
Si dos usuarios tienen comportamientos similares, es razonable suponer que
los productos valorados positivamente por uno también podrían interesar al otro.

## Resultado esperado
Obtener una lista de productos recomendados para un usuario dado.


In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Markdown, display

# =========================
# CARGA
# =========================
df = pd.read_csv("data/processed/reward_data.csv")
products = pd.read_csv("data/raw/products.csv")

# =========================
# MATRIZ USUARIO-PRODUCTO
# =========================
user_item_matrix = df.pivot_table(
    index="user_id",
    columns="product_id",
    values="reward_final",
    aggfunc="mean",
    fill_value=0
)

print("Shape matriz usuario-producto:", user_item_matrix.shape)
display(user_item_matrix.head())

# =========================
# SIMILITUD ENTRE USUARIOS
# =========================
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

display(user_similarity_df.iloc[:5, :5])

# =========================
# FUNCIÓN DE RECOMENDACIÓN
# =========================
def recommend_products(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return pd.DataFrame()

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    similar_users = similar_users.drop(user_id)

    seen_products = set(df[df["user_id"] == user_id]["product_id"].unique())
    candidate_scores = {}

    for similar_user, similarity_score in similar_users.items():
        sim_user_data = df[df["user_id"] == similar_user][["product_id", "reward_final"]]

        for _, row in sim_user_data.iterrows():
            product_id = row["product_id"]

            if product_id not in seen_products:
                candidate_scores[product_id] = candidate_scores.get(product_id, 0) + similarity_score * row["reward_final"]

    recs = pd.DataFrame(
        [{"product_id": k, "score": v} for k, v in candidate_scores.items()]
    )

    if recs.empty:
        return recs

    recs = recs.sort_values("score", ascending=False).head(top_n)
    recs = recs.merge(products, on="product_id", how="left")

    return recs

# =========================
# EJEMPLO
# =========================
sample_user = df["user_id"].iloc[0]
recs = recommend_products(sample_user, top_n=10)

display(Markdown(f"## 🛒 Recomendaciones para el usuario **{sample_user}**"))
display(recs)

Shape matriz usuario-producto: (500, 800)


product_id,1,2,3,4,5,6,7,8,9,10,...,791,792,793,794,795,796,797,798,799,800
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.7,0.0,0.0,0.0,0.0,0.0,0.3,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.7,0.0


user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.031884,0.000000,0.038231,0.000000
2,0.031884,1.000000,0.017814,0.000000,0.018356
3,0.000000,0.017814,1.000000,0.025429,0.000000
4,0.038231,0.000000,0.025429,1.000000,0.000000
5,0.000000,0.018356,0.000000,0.000000,1.000000


## 🛒 Recomendaciones para el usuario **320**

,product_id,score,category,price
0,13.0,0.773817,frutas,94.48
1,392.0,0.693576,bebidas,73.90
2,746.0,0.678300,frutas,49.30
3,194.0,0.593019,limpieza,77.27
4,239.0,0.589357,lacteos,79.86
5,277.0,0.581261,bebidas,4.29
6,216.0,0.569929,bebidas,28.15
7,670.0,0.554309,bebidas,97.59
8,55.0,0.544275,lacteos,24.64
9,103.0,0.543550,frutas,59.70
